# UTILS

In [1]:
def compute_split(y, tile_size, H, n_splits):
    """
    Определяет split по вертикали (по центру тайла)
    """
    center_y = y + tile_size // 2
    split_height = H // n_splits

    split_id = center_y // split_height

    # защита от выхода за границы
    return min(split_id, n_splits - 1)

In [2]:
import numpy as np

def extract_tiles(data, mask, valid, tile_size=128, stride=32):
    """
    data: np.ndarray
        shape: (C, H, W) or (H, W, C)
    mask: (H, W)
    valid: (H, W)

    return: list of dictionaries with tiles
    """

    # Convert to (C, H, W)
    if data.ndim == 2:
        data = data[None, ...]
    elif data.shape[-1] in [1, 3]:
        data = np.transpose(data, (2, 0, 1))  # HWC → CHW

    C, H, W = data.shape

    tiles = []
    tile_id = 0

    for y in range(0, H - tile_size + 1, stride):
        for x in range(0, W - tile_size + 1, stride):

            tiles.append({
                "tile_id": tile_id,
                "y": y,
                "x": x,
                "tile_size": tile_size,

                "data": data[:, y:y+tile_size, x:x+tile_size],
                "mask": mask[y:y+tile_size, x:x+tile_size],
                "valid": valid[y:y+tile_size, x:x+tile_size],
            })

            tile_id += 1

    return tiles

In [3]:
import pandas as pd
import numpy as np

def build_manifest(
    tiles,
    H,
    n_splits=5,
    validation_split=4,
    full_map=False
):
    """
    tiles: список из extract_tiles()
    H: высота исходного изображения
    n_splits: общее количество сплитов
    validation_split: какой сплит использовать как val
    """

    rows = []

    for tile in tiles:
        mask = tile["mask"]
        valid = tile["valid"]

        total_pixels = mask.size
        valid_pixels = np.sum(valid)

        if valid_pixels == 0:
            continue

        positive_pixels = np.sum(mask * valid)

        valid_fraction = valid_pixels / total_pixels
        positive_fraction = positive_pixels / valid_pixels

        # --- вычисляем split ---
        split_id = compute_split(
            tile["y"],
            tile["tile_size"],
            H,
            n_splits
        ) if not full_map else 0

        subset = "full" if full_map else "val" if split_id == validation_split else "train"

        rows.append({
            "tile_id": tile["tile_id"],
            "y": tile["y"],
            "x": tile["x"],
            "tile_size": tile["tile_size"],

            "split_id": int(split_id),
            "subset": subset,

            "valid_fraction": float(valid_fraction),
            "positive_fraction": float(positive_fraction),
            "has_positive": int(positive_pixels > 0)
        })

    return pd.DataFrame(rows)

# Train splits

In [ ]:
n_splits = 4
split_number = 0

data_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/DEM_raw.npz"

In [11]:
import numpy as np

_data = np.load(data_path)
data = _data["data"].astype(np.float32)
valid = _data["valid"].astype(np.uint8)
mask = _data["mask"].astype(np.uint8)

tiles = extract_tiles(data, mask, valid, tile_size=128, stride=32)

H = data.shape[-2] if data.ndim == 3 else data.shape[0]

FileNotFoundError: [Errno 2] No such file or directory: '/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/terasses/DEM(sk)_norm_2.npz'

In [ ]:
manifest = build_manifest(
    tiles,
    H=H,
    n_splits=n_splits,
    validation_split=split_number
)

manifest.to_parquet(
    save_path+f"split{split_number}_manifest.parquet",
    index=False,
    engine="pyarrow"
)

# TEST

In [ ]:
df = pd.read_parquet(save_path+f"split{split_number}_manifest.parquet")

In [ ]:
df["subset"].value_counts()

subset
train    31772
val      13724
Name: count, dtype: int64

In [ ]:
df["split_id"].value_counts()

split_id
0    13724
3    12384
2    11199
1     8189
Name: count, dtype: int64

In [ ]:
df["has_positive"].value_counts(normalize=True)

has_positive
0    0.915971
1    0.084029
Name: proportion, dtype: float64

In [ ]:
df["valid_fraction"].describe()

count    45496.000000
mean         0.975255
std          0.118296
min          0.014160
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: valid_fraction, dtype: float64

# Full split

In [4]:
n_splits = 0
split_number = 0

data_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/raw/DEM21_opt.npz"
save_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/"

In [6]:
import numpy as np

_data = np.load(data_path)
data = _data["dataset"].astype(np.float32)
valid = _data["validMask"].astype(np.uint8)
#mask = _data["mask"].astype(np.float32)
mask = np.zeros_like(valid, np.uint8)

tiles = extract_tiles(data, mask, valid, tile_size=128, stride=32)

H = data.shape[-2] if data.ndim == 3 else data.shape[0]

In [7]:
manifest = build_manifest(
    tiles,
    H=H,
    n_splits=n_splits,
    validation_split=split_number,
    full_map=True
)

manifest.to_parquet(
    save_path+"full_map_manifest.parquet",
    index=False,
    engine="pyarrow"
)

In [8]:
df = pd.read_parquet(save_path+"full_map_manifest.parquet")

In [9]:
df.describe

<bound method NDFrame.describe of         tile_id      y      x  tile_size  split_id subset  valid_fraction  \
0             0      0      0        128         0   full           255.0   
1             1      0     32        128         0   full           255.0   
2             2      0     64        128         0   full           255.0   
3             3      0     96        128         0   full           255.0   
4             4      0    128        128         0   full           255.0   
...         ...    ...    ...        ...       ...    ...             ...   
745555   745555  34848  21728        128         0   full           255.0   
745556   745556  34848  21760        128         0   full           255.0   
745557   745557  34848  21792        128         0   full           255.0   
745558   745558  34848  21824        128         0   full           255.0   
745559   745559  34848  21856        128         0   full           255.0   

        positive_fraction  has_positive  

# Terrases

In [3]:
import os
import numpy as np
import pandas as pd

# -----------------------------------
# CONFIG
# -----------------------------------
tile_size = 512
stride = 128

save_dir = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/terrases"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, f"{tile_size}_{stride}_manifest.parquet")

# Берём по одному репрезентативному npz на локацию.
# Поскольку mask/valid/shape совпадают внутри локации, этого достаточно.
location_npz = {
    "ladzany": "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/terrases/ISO(ladzany)_2.npz",
    "sasovske": "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/terrases/ISO(sasovske)_2.npz",
    "sitno": "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/terrases/ISO(sitno)_2.npz",
    "tribec": "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/terrases/ISO(tribec).npz",
}

# Какие локации участвуют в LOOCV
use_for_loocv_map = {
    "ladzany": 1,
    "sasovske": 1,
    "sitno": 1,
    "tribec": 0,
}

# -----------------------------------
# BUILD MASTER MANIFEST
# -----------------------------------
rows = []
global_tile_id = 0

for location_id, npz_path in location_npz.items():
    print(f"Processing location: {location_id}")
    
    data = np.load(npz_path, allow_pickle=True)
    arr = data["data"].astype(np.float32)     # [C, H, W]
    mask = data["mask"].astype(np.uint8)      # [H, W]
    valid = data["valid"].astype(np.uint8)    # [H, W]

    if arr.ndim != 3:
        raise ValueError(f"{location_id}: expected data shape [C, H, W], got {arr.shape}")

    C, H, W = arr.shape

    if mask.shape != (H, W):
        raise ValueError(f"{location_id}: mask shape {mask.shape} does not match data spatial shape {(H, W)}")

    if valid.shape != (H, W):
        raise ValueError(f"{location_id}: valid shape {valid.shape} does not match data spatial shape {(H, W)}")

    tile_id_local = 0

    for y in range(0, H - tile_size + 1, stride):
        for x in range(0, W - tile_size + 1, stride):
            tile_mask = mask[y:y+tile_size, x:x+tile_size]
            tile_valid = valid[y:y+tile_size, x:x+tile_size]

            total_pixels = tile_size * tile_size
            valid_pixels = int(tile_valid.sum())

            if valid_pixels == 0:
                continue

            positive_pixels = int((tile_mask * tile_valid).sum())

            valid_fraction = valid_pixels / total_pixels
            positive_fraction = positive_pixels / valid_pixels

            rows.append({
                "global_tile_id": global_tile_id,
                "tile_id": tile_id_local,

                "location_id": location_id,
                "npz_path_used_for_grid": npz_path,
                "use_for_loocv": int(use_for_loocv_map[location_id]),

                "y": int(y),
                "x": int(x),
                "tile_size": int(tile_size),
                "stride": int(stride),

                "height": int(H),
                "width": int(W),
                "channels": int(C),

                "total_pixels": int(total_pixels),
                "valid_pixels": int(valid_pixels),
                "positive_pixels": int(positive_pixels),

                "valid_fraction": float(valid_fraction),
                "positive_fraction": float(positive_fraction),
                "has_positive": int(positive_pixels > 0),
            })

            tile_id_local += 1
            global_tile_id += 1

manifest = pd.DataFrame(rows)

# -----------------------------------
# OPTIONAL SORT
# -----------------------------------
manifest = manifest.sort_values(
    by=["location_id", "y", "x"],
    ascending=[True, True, True]
).reset_index(drop=True)

# -----------------------------------
# SAVE
# -----------------------------------
manifest.to_parquet(save_path, index=False, engine="pyarrow")

print("\nSaved manifest:")
print(save_path)
print("\nShape:", manifest.shape)

print("\nRows per location:")
print(manifest.groupby("location_id").size())

print("\nPositive tiles per location:")
print(manifest.groupby("location_id")["has_positive"].sum())

print("\nValid fraction stats:")
print(manifest["valid_fraction"].describe())

print("\nPositive fraction stats:")
print(manifest["positive_fraction"].describe())

manifest.head()

Processing location: ladzany
Processing location: sasovske
Processing location: sitno
Processing location: tribec

Saved manifest:
/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/data/processed/terrases/512_128_manifest.parquet

Shape: (19991, 18)

Rows per location:
location_id
ladzany       741
sasovske      378
sitno        1092
tribec      17780
dtype: int64

Positive tiles per location:
location_id
ladzany     133
sasovske    135
sitno       380
tribec        0
Name: has_positive, dtype: int64

Valid fraction stats:
count    19991.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: valid_fraction, dtype: float64

Positive fraction stats:
count    19991.000000
mean         0.001427
std          0.011831
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.239166
Name: positive_fraction, dtype: float64


,global_tile_id,tile_id,location_id,npz_path_used_for_grid,use_for_loocv,y,x,tile_size,stride,height,width,channels,total_pixels,valid_pixels,positive_pixels,valid_fraction,positive_fraction,has_positive
0,0,0,ladzany,/home/nikitachernysh/Storage/Projects/lidar-ar...,1,0,0,512,128,5411,2895,3,262144,262144,0,1.0,0.0,0
1,1,1,ladzany,/home/nikitachernysh/Storage/Projects/lidar-ar...,1,0,128,512,128,5411,2895,3,262144,262144,0,1.0,0.0,0
2,2,2,ladzany,/home/nikitachernysh/Storage/Projects/lidar-ar...,1,0,256,512,128,5411,2895,3,262144,262144,0,1.0,0.0,0
3,3,3,ladzany,/home/nikitachernysh/Storage/Projects/lidar-ar...,1,0,384,512,128,5411,2895,3,262144,262144,0,1.0,0.0,0
4,4,4,ladzany,/home/nikitachernysh/Storage/Projects/lidar-ar...,1,0,512,512,128,5411,2895,3,262144,262144,0,1.0,0.0,0
